In [ ]:
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd
import ast
import tiktoken
import json
from openai import OpenAI
import re
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from itertools import accumulate
import math
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

In [3]:
load_dotenv(override=True)

True

In [42]:
OUTLIER_EXECUTED = False

In [ ]:
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Musical_Instruments", split="full", trust_remote_code=True)

print("Number of musical instrument reviews:", len(dataset))

raw/meta_categories/meta_Musical_Instrum(…):   0%|          | 0.00/632M [00:00<?, ?B/s]

Generating full split:   0%|          | 0/213593 [00:00<?, ? examples/s]

{'main_category': 'Musical Instruments', 'title': 'Pearl Export Lacquer EXL725S/C249 5-Piece New Fusion Drum Set with Hardware, Honey Amber', 'average_rating': 4.2, 'rating_number': 22, 'features': ['Item may ship in more than one box and may arrive separately', '(22x18, 10x7, 12x8, 16x16, 14x5.5)', 'P930 Demonator Pedal', '830 Hardware Pack', 'Matching snare, REMO snare batter side head'], 'description': ["Introducing the best selling drum set of all time... Export Series returns and this time with a lacquer finish. EXL Export Lacquer Series incorporates Pearl's S.S.T. Superior Shell Technology, Opti-Loc tom mounts, all-new 830 Series Hardware with a P-930 Pedal, and a choice of three amazing stocking finishes."], 'price': 'None', 'images': {'hi_res': ['https://m.media-amazon.com/images/I/91RuLqvx9IL._AC_SL1500_.jpg', 'https://m.media-amazon.com/images/I/81q8vubRs-L._AC_SL1500_.jpg', None, 'https://m.media-amazon.com/images/I/81ubSuvhnrL._AC_SL1500_.jpg', 'https://m.media-amazon.com/i

In [5]:
print(dataset[6])

{'main_category': 'Musical Instruments', 'title': 'Fender Classic Series Case Stand, 3-Guitar, Black', 'average_rating': 4.5, 'rating_number': 80, 'features': ['Tweed or black vinyl-wrapped 3-ply hard-shell wooden case', 'Vinyl-wrapped, steel-core carry handle', 'Crushed-acrylic plush interior lining', 'Steel latches', 'Can hold up to 3 instruments'], 'description': ['Product Description', 'Fender Instrument Case Stands are a great way to display and protect multiple instruments. This case stand looks like a traditional case but easily turns into a stage-worthy guitar stand. Crafted with road-reliable materials, this 3-ply hard-shell wooden case boasts a vinyl-wrapped steel carry handle and steel latches. The soft, crushed-acrylic plush interior lining ensures your guitars remain scratch and damage-free.', 'From the Manufacturer', 'Fender Instrument Case Stands are a great way to display and protect multiple instruments. This case stand looks like a traditional case but easily turns in

In [43]:
data = pd.DataFrame(dataset, columns=["title", "description", "features", "details", "price"])

In [44]:
data["title"] = data["title"].apply(str)
data["description"] = data["description"].apply(str)
data["features"] = data["features"].apply(str)

data["price"] = data["price"].replace(["—", "-", "", "None"], None)
data["title"] = data["title"].replace("", None)
data["description"] = data["description"].replace("[]", None)
data["features"] = data["features"].replace("[]", None)

print("Number of musical instrument reviews before cleaning:", len(data))

Number of musical instrument reviews before cleaning: 213593


In [47]:
data["price"] = pd.to_numeric(data["price"], errors="coerce")

data = data.dropna()

data["price"] = data["price"].apply(float)

print("Number of musical instrument reviews before removing duplicates:", len(data))

data = data.drop_duplicates(subset=["title", "description", "price"])

Number of musical instrument reviews before removing duplicates: 60733


In [48]:
print("Number of musical instrument reviews after removing duplicates:", len(data))

Number of musical instrument reviews after removing duplicates: 60733


In [49]:
q1 = data["price"].quantile(0.25)
q3 = data["price"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

if not OUTLIER_EXECUTED:
  print("Removing outliers that fall outside the range of", lower_bound, "to", upper_bound)
  data = data[(data["price"] >= lower_bound) & (data["price"] <= upper_bound)]
  OUTLIER_EXECUTED = True
  print("Number of musical instrument reviews after removing outliers:", len(data))
else:
  print("Outlier removal has already been executed. Skipping this step.")

Removing outliers that fall outside the range of -157.61 to 302.55
Number of musical instrument reviews after removing outliers: 54185


In [ ]:
def clean_list_string(list_string):
    """Convert string representation of list to clean string"""
    try:
        if list_string.startswith("[") and list_string.endswith("]"):
            parsed = ast.literal_eval(list_string)
            return " ".join(str(item) for item in parsed)
    except:
        pass
    return str(list_string)

def clean_dict_string(dict_string):
    """Convert string representation of dict to clean string"""
    try:
        if dict_string.startswith("{") and dict_string.endswith("}"):
            parsed = ast.literal_eval(dict_string)
            parts = []
            for key, value in parsed.items():
                if isinstance(value, dict):
                    value = ", ".join(f"{k}: {v}" for k, v in value.items())
                parts.append(f"{key}: {value}")
            return " | ".join(parts)
    except:
        pass
    return str(dict_string)

data["description"] = data["description"].apply(clean_list_string)
data["features"] = data["features"].apply(clean_list_string)
data["details"] = data["details"].apply(clean_dict_string)

Sample cleaned review: title          VocoPro, plug in, Black, 21.00 x 21.00 x 23.00...
description    ['VocoPro UHF-18 DIAMOND - N Wireless Micropho...
features       ['Includes one microphone and one receiver', '...
details        {"Item Weight": "2.29 pounds", "Product Dimens...
price                                                      112.0
Name: 3, dtype: object


In [51]:
SYSTEM_PROMPT = """
You are a price prediction expert. Given a product's title, description, features, or details, predict its price in USD.

Rules:
1. Analyze all available product information carefully
2. If information is incomplete or truncated, use your knowledge of similar products and market pricing to make informed predictions
3. Consider product quality indicators, brand reputation, features, and typical market values
4. Return ONLY the numeric price (e.g., "29.99") 
5. Do not include currency symbols, explanations, or additional text 
6. Return just the raw float number
"""

In [53]:
def truncate_by_tokens(text, max_tokens=300):
  """Truncate to max tokens"""
  encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
  tokens = encoding.encode(text)

  if len(tokens) <= max_tokens:
    return text

  truncated_tokens = tokens[:max_tokens]
  return encoding.decode(truncated_tokens)

def generate_prompt(data):
  """
  Generate a prompt for the model to predict the price of a product
  """

  prompt = f"""
  Below are the details of the product:
  Title: {data['title']}
  Description: {data['description']}
  Features: {data['features']}
  """
  return truncate_by_tokens(prompt)

def generate_message(data):
  """
  Generate a message for the model to predict the price of a product
  """
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": data["prompt"]},
      {"role": "assistant", "content": str(data['price'])}
  ]
  return messages


In [54]:
data["prompt"] = data.apply(lambda x: generate_prompt(x), axis=1)

In [55]:
train_data = data.sample(n=500, random_state=42)
train_set = train_data.sample(frac=0.8, random_state=42)
validation_set = train_data.drop(train_set.index)

In [57]:
with open("train.jsonl", "w") as file:
  for index, row in train_set.iterrows():
    messages = {"messages": generate_message(row)}
    file.write(json.dumps(messages) + "\n")

with open("validation.jsonl", "w") as file:
  for index, row in validation_set.iterrows():
    messages = {"messages": generate_message(row)}
    file.write(json.dumps(messages) + "\n")

In [59]:
openai = OpenAI()

In [60]:
print("Uploading training file...")
training_file = openai.files.create(
    file=open('train.jsonl', 'rb'),
    purpose='fine-tune'
)
print(f"File uploaded: {training_file.id}")

print("Uploading validation file...")
validation_file = openai.files.create(
    file=open('validation.jsonl', 'rb'),
    purpose='fine-tune'
)
print(f"Validation file uploaded: {validation_file.id}")

Uploading training file...
File uploaded: file-HVohmsXyq9xSFWbiKG6Nds
Uploading validation file...
Validation file uploaded: file-Fh4tKTZSQUqvTwVqb3WS1d
Starting fine-tuning...
Job created: ftjob-Rfd6ow3WN1HDBUuvbBT9zDMa
Status: validating_files
Status: validating_files
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: succeeded
Model ready: ft:gpt-4.1-mini-2025-04-14:personal::DFo1SSNk


In [ ]:
# Run this to train the model. This can take 10-20 minutes to complete, so be patient!
import time

print("Starting fine-tuning...")
job = openai.fine_tuning.jobs.create(
    validation_file=validation_file.id,
    training_file=training_file.id,
    model='gpt-4.1-mini-2025-04-14'
)
print(f"Job created: {job.id}")

status = openai.fine_tuning.jobs.retrieve(job.id)
print(f"Status: {status.status}")

while status.status not in ['succeeded', 'failed']:
    time.sleep(60)
    status = openai.fine_tuning.jobs.retrieve(job.id)
    print(f"Status: {status.status}")

if status.status == 'succeeded':
    print(f"Model ready: {status.fine_tuned_model}")
else:
    print(f"Training failed: {status.error}")

In [69]:
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

WORKERS = 1
DEFAULT_SIZE = 100


class Tester:
    def __init__(self, predictor, data, title=None, size=DEFAULT_SIZE, workers=WORKERS):
        self.predictor = predictor
        self.data = data
        self.title = title or self.make_title(predictor)
        self.size = size
        self.titles = []
        self.guesses = []
        self.truths = []
        self.errors = []
        self.colors = []
        self.workers = workers

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title().replace("Gpt", "GPT")

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        else:
            return value

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:
            return "green"
        elif error < 80 or error / truth < 0.4:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value = self.predictor(datapoint)
        guess = self.post_process(value)
        truth = datapoint["price"]
        error = abs(guess - truth)
        color = self.color_for(error, truth)
        title = datapoint["title"] if len(datapoint["title"]) <= 40 else datapoint["title"][:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame(
            {
                "truth": self.truths,
                "guess": self.guesses,
                "title": self.titles,
                "error": self.errors,
                "color": self.colors,
            }
        )

        # Pre-format hover text
        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]

        max_val = float(max(df["truth"].max(), df["guess"].max()))

        fig = px.scatter(
            df,
            x="truth",
            y="guess",
            color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title,
            labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=1000,
            height=800,
        )

        # Assign customdata per trace (one color/category = one trace)
        for tr in fig.data:
            mask = df["color"] == tr.name
            tr.customdata = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)

        # Reference line y=x
        fig.add_trace(
            go.Scatter(
                x=[0, max_val],
                y=[0, max_val],
                mode="lines",
                line=dict(width=2, dash="dash", color="deepskyblue"),
                name="y = x",
                hoverinfo="skip",
                showlegend=False,
            )
        )

        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n = len(self.errors)

        # Running mean and std (pure Python)
        running_sums = list(accumulate(self.errors))
        x = list(range(1, n + 1))
        running_means = [s / i for s, i in zip(running_sums, x)]

        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds = [
            math.sqrt((sq_sum / i) - (mean**2)) if i > 1 else 0
            for i, sq_sum, mean in zip(x, running_squares, running_means)
        ]

        # 95% confidence interval for mean
        ci = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]

        # Plot
        fig = go.Figure()

        # Shaded confidence interval band
        fig.add_trace(
            go.Scatter(
                x=x + x[::-1],
                y=upper + lower[::-1],
                fill="toself",
                fillcolor="rgba(128,128,128,0.2)",
                line=dict(color="rgba(255,255,255,0)"),
                hoverinfo="skip",
                showlegend=False,
                name="95% CI",
            )
        )

        # Main line with hover text showing CI
        fig.add_trace(
            go.Scatter(
                x=x,
                y=running_means,
                mode="lines",
                line=dict(width=3, color="firebrick"),
                name="Cumulative Avg Error",
                customdata=list(
                    zip(
                        ci,
                    )
                ),
                hovertemplate=(
                    "n=%{x}<br>"
                    "Avg Error=$%{y:,.2f}<br>"
                    "±95% CI=$%{customdata[0]:,.2f}<extra></extra>"
                ),
            )
        )

        # Title with final stats
        final_mean = running_means[-1]
        final_ci = ci[-1]
        title = f"{self.title} Error: ${final_mean:,.2f} ± ${final_ci:,.2f}"

        fig.update_layout(
            title=title,
            xaxis_title="Number of Datapoints",
            yaxis_title="Average Absolute Error ($)",
            width=1000,
            height=360,
            template="plotly_white",
            showlegend=False,
        )

        fig.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        mse = mean_squared_error(self.truths, self.guesses)
        r2 = r2_score(self.truths, self.guesses) * 100
        title = f"{self.title} results<br><b>Error:</b> ${average_error:,.2f} <b>MSE:</b> {mse:,.0f} <b>r²:</b> {r2:.1f}%"
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        with ThreadPoolExecutor(max_workers=self.workers) as ex:
            for title, guess, truth, error, color in tqdm(
                ex.map(self.run_datapoint, range(self.size)), total=self.size
            ):
                self.titles.append(title)
                self.guesses.append(guess)
                self.truths.append(truth)
                self.errors.append(error)
                self.colors.append(color)
                print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        self.report()


def evaluate(function, data, size=DEFAULT_SIZE, workers=WORKERS):
    Tester(function, data, size=size, workers=workers).run()

In [ ]:
FINE_TUNED_MODEL = "ft:gpt-4.1-mini-2025-04-14:personal::DFo1SSNk"

In [ ]:
def predictor(data):
    user_prompt = data["description"]
    if not user_prompt or user_prompt.strip() == "":
        print("Warning: Empty prompt!")
        return data["price"]

    user_prompt = f"""
    Return the price of the product in USD.
    Return just the raw float number.

    Product Description: {user_prompt}
    Note: Numbers in this description show product specifications like:
    - Dimensions (size measurements)
    - Weight (ounces/pounds)
    - Rankings (popularity/sales rank)
    - Part/model numbers

    Price prediction:
    """

    test = openai.chat.completions.create(
        # uncomment this line to use your own model
        # model=status.fine_tuned_model,
        model=FINE_TUNED_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    result = test.choices[0].message.content
    return result

In [64]:
train_titles = set(train_set["title"])
validation_titles = set(validation_set["title"])
exclude_titles = train_titles.union(validation_titles)

test_candidates = data[~data["title"].isin(exclude_titles)]

sample_size = 100
test_sample = test_candidates.sample(n=sample_size, random_state=42)

In [66]:
test_data = test_sample.to_dict(orient="records")

In [68]:
print(test_data[0])

{'title': 'Shofar Zion Smooth Mouthpiece for Easy Blowing | Made in Israel | Includes Exquisite Stand and Carrying Bag (Yem-Large)', 'description': "['Genuine Kudu horn- This instrument resembles a trumpet however it is all-natural made from Genuine kudu’s horn. Each Shofar is handmade, the production is performed using old and traditional techniques combined with modern techniques, all under very strict supervision to ensure the quality and kashruth of the shofars.']", 'features': '[\'GENUINE KUDU HORN- This shofar is all-natural made from genuine Kudu horn. The shofar or Kudu horn is an ancient Instrument, known to produce a mysterious mystical sound. Traditionally used in Yemenite culture, this horn will blow you away with its unique beauty. Shofar Size is between 32"-36".\', \'SHOFAR ZION QUALITY: Shofar Zion is known to produce top rated shofars, with beauty and quality surpassing all others on the market. Production is under strict supervision to ensure complete kashruth. All sho

In [70]:
evaluate(predictor, test_data, size=sample_size, workers=WORKERS)

  0%|          | 0/100 [00:00<?, ?it/s]

$100 $2 $93 $13 $0 $64 $127 $8 $27 $5 $4 $32 $2 $9 $10 $10 $6 $6 $2 $5 $32 $0 $138 $9 $10 $23 $5 $3 $28 $21 $7 $7 $29 $4 $23 $0 $19 $23 $20 $1 $47 $26 $30 $1 $2 $3 $159 $15 $8 $104 $50 $7 $47 $4 $3 $150 $0 $2 $6 $7 $1 $7 $19 $39 $9 $0 $17 $0 $12 $54 $83 $4 $24 $4 $0 $1 $5 $4 $1 $5 $14 $10 $0 $140 $5 $12 $28 $3 $3 $29 $2 $7 $6 $3 $20 $31 $5 $2 $4 $23 